In [1]:
import requests
import os

YOUR_USER_ID = os.getenv("LEETGPU_USER_ID")# login into leetgpu and inspect request to find x-user-id header

headers = {
    "accept": "*/*",
    "accept-language": "en-US,en;q=0.9,ru;q=0.8",
    "cache-control": "no-cache",
    "pragma": "no-cache",
    "priority": "u=1, i",
    "sec-ch-ua": "\"Chromium\";v=\"136\", \"Google Chrome\";v=\"136\", \"Not.A/Brand\";v=\"99\"",
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": "\"Windows\"",
    "sec-fetch-dest": "empty",
    "sec-fetch-mode": "cors",
    "sec-fetch-site": "same-site",
    "x-user-id": YOUR_USER_ID
}

response = requests.get("https://api.leetgpu.com/api/v1/challenges/fetch-all", headers=headers)
challenges = response.json()

In [4]:
problem_names = {}
for challenge in challenges:
    id = challenge['id']
    title = challenge['title']

    title = title.lower().replace(' ', '_').replace('-', '_').replace('(', '_').replace(')', '_').replace('&', '_')
    while '__' in title: title = title.replace('__', '_')
    if title.endswith('_'):
        title = title[:-1]
    problem_names[id] = title
    
def find_lowest_runtime_code(submission_data,language, gpu_type):
    if language in submission_data and gpu_type in submission_data[language]:
        return min(submission_data[language][gpu_type], key=lambda x: x['runtime'])
    return None

problem_set = {}

for id,problem_name in problem_names.items():
    submissions = requests.get(f"https://api.leetgpu.com/api/v1/challenges/{id}/submissions", headers=headers)
    submissions_json = submissions.json()
    if "msg" in submissions_json:
        print(f"Error fetching submissions for {problem_name}: {submissions_json}")
        exit()
    problem_set[id] = (problem_name,submissions_json)


In [ ]:
for id,problem in problem_set.items():
    (problem_name,submissions) = problem
    #print(problem_name)
    submission_data = {}
    for submission in submissions:
        if submission['status'] == "SUCCESS":
            runtime = submission['runtime']
            code = submission['code']
            language = submission['language']
            gpu_type = submission["gpu"] 

            if language not in submission_data:
                submission_data[language] = {}
            if gpu_type not in submission_data[language]:
                submission_data[language][gpu_type] = []

            submission_data[language][gpu_type].append({'runtime': runtime, 'code': code})

    best_submission = find_lowest_runtime_code(submission_data, "cuda","NVIDIA TESLA T4")
    if best_submission is not None:
        with open(f"{id}.{problem_name}.cu", "w") as file:
            problem_url_name = problem_name.replace('_', '-')
            print(f"// URL: https://leetgpu.com/challenges/{problem_url_name}", file=file)
            print("// GPU: NVIDIA TESLA T4", file=file)
            print("// Runtime: "+str(best_submission['runtime'])+" ms", file=file)
            print(best_submission['code'], file=file)
        print(problem_name,best_submission)
        